# convex-swaghaus-bindle — DX walkthrough

Provider-neutral Bindle: `b.web(convexApp(root))` + `b.data()` + WorkOS `b.external`.
Matching selects `@hobo/provider-convex` at Yard — Bindle never pins convex.

In [1]:
import { bindle } from "@hobo/bindles";
import {
  convexApp,
  convexRootFromAnnotations,
  isConvexWebAppAnnotation,
} from "@hobo/convex";

const definition = bindle((b) => {
  const backend = b.data();
  const auth = b.external({ endpoints: { workos: "https://api.workos.com" } });
  const web = b.web(convexApp("examples/convex-swaghaus")).dependsOn(backend, auth);
  return { backend, auth, web };
});

const root = convexRootFromAnnotations(definition.resources.web!.annotations);
const convex = definition.resources.web!.annotations.some(isConvexWebAppAnnotation);
console.log(`root=${root} convex=${convex}`);

root=examples/convex-swaghaus convex=true


## Parse boundary

The serialized manifest crosses `src/parse.ts` once and becomes a refined
plan with every required env name routed. A provider pin on the Bindle is
refused at authoring time, with the Camp/Yard remediation attached.

In [2]:
import {
  authorSwaghausBindle,
  draftSwaghausManifest,
  parseSwaghausBindle,
} from "./src/index.ts";

const { plan } = authorSwaghausBindle({ root: "examples/convex-swaghaus" });
console.log(`${plan.appName} ${plan.framework} ${plan.webRoot}`);
console.log(plan.envDeps.map((dep) => `${dep.name}=${dep.routing}`).join(" "));

const pinned = draftSwaghausManifest({ root: "examples/convex-swaghaus" });
(pinned.resources.web as { annotations: unknown[] }).annotations.push({
  provider: "convex",
  option: "deployment",
  config: { region: "us-east" },
});
try {
  parseSwaghausBindle(pinned);
} catch (error) {
  console.log(`refused ${(error as Error).message}`);
}

app convex examples/convex-swaghaus
CONVEX_DEPLOY_KEY=env VITE_CONVEX_URL=data VITE_WORKOS_CLIENT_ID=external VITE_WORKOS_REDIRECT_URI=external WORKOS_CLIENT_ID=external
refused web.annotations[]: provider pin "convex" is not allowed on a Bindle — scope provider options to a Camp or Yard
